# R3-N1 Stage 2: Operational Calibration

Fits the Stage 1 Platt and Smoothed Isotonic (PCHIP, K=10) methods on the 960 calibration recordings, then evaluates frozen mappings on the separate 1,040-recording operating partition. Operating labels are used only for evaluation and threshold selection. The test partition is never loaded. No BiGRU inference or training is performed.

Source lineage: `runs/r3/r3_m1_20261107T144037Z/`. Expected local hashes are pinned below. The selected checkpoint is hash-checked as an unchanged artifact; it is never loaded.


In [ ]:
import hashlib, json, os, sys, time
from datetime import datetime, timezone
from pathlib import Path

# Mount Drive in Colab when the project root is not already available.
if not Path('/content/drive/MyDrive/PAI&AIES Project/src/echoproof').is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

# Colab root with local-workspace fallbacks.
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
if not (ROOT / 'src/echoproof').is_dir():
    candidates = [Path(os.environ.get('ECHOPROOF_ROOT', '')), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if p and (p / 'src/echoproof').is_dir()), ROOT)
assert (ROOT / 'src/echoproof').is_dir(), f'EchoProof source tree not found under {ROOT}'
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit
from sklearn.isotonic import IsotonicRegression

from echoproof.artifacts import create_run, file_digest, make_manifest, write_json
from echoproof.evaluation.calibrate import DEFAULT_EPSILON
from echoproof.evaluation.metrics import _compute_roc_auc, interpolated_eer
from echoproof.evaluation.threshold import select_threshold

SOURCE_DIR = ROOT / 'runs/r3/r3_m1_20261107T144037Z'
CAL_FILE = SOURCE_DIR / 'predictions_recordings_calibration.parquet'
OPER_FILE = SOURCE_DIR / 'predictions_recordings_operating.parquet'
CHECKPOINT_FILE = SOURCE_DIR / 'r3_m1_20261107T144037Z-lower_lr/best.pt'
SELECTION_FILE = SOURCE_DIR / 'selection.json'
EVIDENCE_FILE = SOURCE_DIR / 'evidence.json'
CONTRACT_FILE = SOURCE_DIR / 'experiment_contract.json'
EXPECTED = {
    'calibration': '2a1b6aa6b8bcbcca578a047ea321210b755a5de5ff815e1705a7b1d9374d26c3',
    'operating': '73af6567574094fee269aeabdcba218c78fee66383e9836ccea0db5a65e10f31',
    'checkpoint': '3154da86d9faf603c4f8bc5bf7abefdf5aa5c8fd6fc2e3aebe12324df194e640',
}
EXPECTED_SCHEMA = ['recording_id','group_id','label','component','validation_role','raw_score','chunk_count']

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

# Fail before reading prediction tables or creating output if lineage hashes drift.
for name, path in [('calibration', CAL_FILE), ('operating', OPER_FILE), ('checkpoint', CHECKPOINT_FILE)]:
    assert path.is_file(), f'Required lineage artifact missing: {path}'
    actual = sha256(path)
    assert actual == EXPECTED[name], f'{name} SHA-256 mismatch: expected {EXPECTED[name]}, got {actual}'
selection = json.loads(SELECTION_FILE.read_text(encoding='utf-8'))
assert selection['selected'] == 'lower_lr' and selection['checkpoint_sha256']['lower_lr'] == EXPECTED['checkpoint']
evidence = json.loads(EVIDENCE_FILE.read_text(encoding='utf-8'))
assert evidence['output_sha256']['predictions_recordings_calibration.parquet'] == EXPECTED['calibration']
assert evidence['output_sha256']['predictions_recordings_operating.parquet'] == EXPECTED['operating']
assert evidence['selection']['selected'] == 'lower_lr'
assert sha256(CONTRACT_FILE) == selection['contract_sha256']
print('Lineage hashes verified. BiGRU checkpoint will not be loaded; test data will not be accessed.')


## Load and validate the two permitted validation roles

Only the named calibration and operating prediction parquet files are read. Each must match its frozen schema, role, row count, ID uniqueness, class support, finite score range, and disjoint recording IDs.


In [ ]:
cal_df = pq.read_table(CAL_FILE).to_pandas()
oper_df = pq.read_table(OPER_FILE).to_pandas()
for name, df, role, count in [('calibration', cal_df, 'calibration', 960), ('operating', oper_df, 'operating', 1040)]:
    assert df.columns.tolist() == EXPECTED_SCHEMA, f'{name} schema mismatch: {df.columns.tolist()}'
    assert len(df) == count, f'{name} count mismatch: {len(df)} != {count}'
    assert df['recording_id'].notna().all() and df['recording_id'].is_unique, f'{name} IDs missing or duplicated'
    assert set(df['validation_role'].astype(str)) == {role}, f'{name} role mismatch'
    assert set(df['label'].unique()) == {0, 1}, f'{name} must contain both labels'
    s = df['raw_score'].to_numpy(dtype=np.float64)
    assert np.isfinite(s).all() and ((s >= 0) & (s <= 1)).all(), f'{name} raw scores invalid'
assert set(cal_df.recording_id.astype(str)).isdisjoint(set(oper_df.recording_id.astype(str))), 'Calibration/operating recording IDs overlap'
assert set(cal_df.group_id.astype(str)).isdisjoint(set(oper_df.group_id.astype(str))), 'Calibration/operating speaker groups overlap'
assert len(cal_df) == 960 and len(oper_df) == 1040
print('Validated 960 calibration and 1040 operating recordings; recording IDs are disjoint.')


## Stage 1 metrics and score transformations

The following logit/sigmoid helpers, epsilon (`1e-7`), log-loss/Brier/ECE functions, and fitting functions are copied from the Stage 1 notebook. The Platt optimizer and Smoothed Isotonic knot construction are reused without changes.


In [ ]:
from echoproof.evaluation.calibrate import DEFAULT_EPSILON, apply_temperature
from scipy.special import expit

def logit(p: np.ndarray, eps: float = DEFAULT_EPSILON) -> np.ndarray:
    p_clip = np.clip(p, eps, 1.0 - eps)
    return np.log(p_clip / (1.0 - p_clip))

def sigmoid(z: np.ndarray) -> np.ndarray:
    return expit(np.asarray(z, dtype=np.float64))

def fit_temperature(labels, scores):
    """Notebook-local temperature fit with stable softplus cross-entropy."""
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    assert set(np.unique(labels)) == {0, 1}, "Temperature fit requires both classes"
    logits = logit(scores)
    signed_labels = 1.0 - 2.0 * labels
    result = minimize_scalar(
        lambda temperature: float(np.mean(np.logaddexp(0.0, signed_labels * logits / temperature))),
        bounds=(0.01, 50.0), method="bounded"
    )
    assert result.success and result.x > 0, f"Temperature optimization failed: {result.message}"
    return float(result.x)

def compute_bce_loss(labels: np.ndarray, scores: np.ndarray, eps: float = DEFAULT_EPSILON) -> float:
    s_clipped = np.clip(scores, eps, 1.0 - eps)
    return float(-np.mean(labels * np.log(s_clipped) + (1.0 - labels) * np.log(1.0 - s_clipped)))

def compute_brier_score(labels: np.ndarray, scores: np.ndarray) -> float:
    return float(np.mean((scores - labels) ** 2))

def compute_equal_width_ece(labels: np.ndarray, scores: np.ndarray, n_bins: int = 10) -> tuple[float, list[dict], int]:
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(labels)
    weighted_err = 0.0
    bins_info = []
    
    for i in range(n_bins):
        low, high = bin_edges[i], bin_edges[i + 1]
        mask = (scores >= low) & (scores <= high) if i == n_bins - 1 else (scores >= low) & (scores < high)
        cnt = int(np.sum(mask))
        if cnt > 0:
            conf = float(np.mean(scores[mask]))
            acc = float(np.mean(labels[mask]))
            err = abs(acc - conf)
            weighted_err += (cnt / total) * err
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_width',
                'bin_range': f"[{low:.1f}, {high:.1f}]" if i == n_bins - 1 else f"[{low:.1f}, {high:.1f})",
                'low': float(low),
                'high': float(high),
                'count': cnt,
                'confidence': conf,
                'accuracy': acc,
                'calibration_error': err,
                'is_empty': False,
            })
        else:
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_width',
                'bin_range': f"[{low:.1f}, {high:.1f}]" if i == n_bins - 1 else f"[{low:.1f}, {high:.1f})",
                'low': float(low),
                'high': float(high),
                'count': 0,
                'confidence': None,
                'accuracy': None,
                'calibration_error': None,
                'is_empty': True,
            })
    ece = float(weighted_err)
    non_empty_bins = sum(1 for b in bins_info if not b['is_empty'])
    return ece, bins_info, non_empty_bins

def compute_equal_mass_ece(labels: np.ndarray, scores: np.ndarray, n_bins: int = 10) -> tuple[float, list[dict], int]:
    total = len(labels)
    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    raw_edges = np.quantile(scores, quantiles)
    unique_edges = np.unique(raw_edges)
    actual_bins = len(unique_edges) - 1
    
    weighted_err = 0.0
    bins_info = []
    
    for i in range(actual_bins):
        low, high = unique_edges[i], unique_edges[i + 1]
        mask = (scores >= low) & (scores <= high) if i == actual_bins - 1 else (scores >= low) & (scores < high)
        cnt = int(np.sum(mask))
        if cnt > 0:
            conf = float(np.mean(scores[mask]))
            acc = float(np.mean(labels[mask]))
            err = abs(acc - conf)
            weighted_err += (cnt / total) * err
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_mass',
                'bin_range': f"[{low:.4f}, {high:.4f}]" if i == actual_bins - 1 else f"[{low:.4f}, {high:.4f})",
                'low': float(low),
                'high': float(high),
                'count': cnt,
                'confidence': conf,
                'accuracy': acc,
                'calibration_error': err,
                'is_empty': False,
            })
        else:
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_mass',
                'bin_range': f"[{low:.4f}, {high:.4f}]" if i == actual_bins - 1 else f"[{low:.4f}, {high:.4f})",
                'low': float(low),
                'high': float(high),
                'count': 0,
                'confidence': None,
                'accuracy': None,
                'calibration_error': None,
                'is_empty': True,
            })
    ece = float(weighted_err)
    non_empty_bins = sum(1 for b in bins_info if not b['is_empty'])
    return ece, bins_info, non_empty_bins

def check_non_decreasing(raw: np.ndarray, cal: np.ndarray) -> bool:
    sorted_idx = np.argsort(raw)
    diffs = np.diff(cal[sorted_idx])
    return bool(np.all(diffs >= -1e-12))

def calculate_metrics(labels, scores):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    assert set(np.unique(labels)) == {0, 1}, "Log Loss, ROC-AUC, and EER require both classes"
    assert np.all(np.isfinite(scores)) and np.all((scores >= 0.0) & (scores <= 1.0)), "Metrics require finite probabilities in [0,1]"
    ece_width, _, _ = compute_equal_width_ece(labels, scores, n_bins=10)
    ece_mass, _, _ = compute_equal_mass_ece(labels, scores, n_bins=10)
    return {"log_loss": compute_bce_loss(labels, scores), "brier_score": compute_brier_score(labels, scores),
            "equal_width_ece": ece_width, "equal_mass_ece": ece_mass,
            "roc_auc": _compute_roc_auc(labels, scores), "eer": interpolated_eer(labels, scores)}

y_true = cal_df['label'].values.astype(np.int64)
s_raw = cal_df['raw_score'].values.astype(np.float64)
groups_arr = cal_df['group_id'].values

print("Authoritative metrics and binning utilities loaded.")

In [ ]:
def fit_platt_model(s_train: np.ndarray, y_train: np.ndarray) -> tuple[float, float]:
    z_tr = logit(s_train)
    t_init = fit_temperature(y_train.tolist(), s_train.tolist())
    
    def platt_loss(params: tuple[float, float]) -> float:
        a, b = params
        z_scaled = a * z_tr + b
        margin = (1.0 - 2.0 * y_train) * z_scaled
        loss = np.logaddexp(0.0, margin)
        return float(np.mean(loss))
        
    res = minimize(platt_loss, x0=[1.0 / t_init, 0.0], bounds=[(1e-4, 50.0), (-20.0, 20.0)], method='L-BFGS-B')
    assert res.success, f"Platt optimization failed: {res.message}"
    a_val, b_val = float(res.x[0]), float(res.x[1])
    assert a_val > 0.0, f"Platt slope must be positive, got {a_val}"
    return a_val, b_val

def apply_platt_model(s_eval: np.ndarray, a: float, b: float) -> np.ndarray:
    z_eval = logit(s_eval)
    return sigmoid(a * z_eval + b)

def fit_smoothed_pchip(s_train: np.ndarray, iso_model: IsotonicRegression, k_knots: int) -> PchipInterpolator:
    q_knots = np.linspace(0.0, 1.0, k_knots)
    x_k = np.unique(np.concatenate([[0.0], np.quantile(s_train, q_knots), [1.0]]))
    y_k = iso_model.predict(x_k)
    assert np.all(np.diff(y_k) >= -1e-12), "Isotonic knot predictions must be non-decreasing"
    return PchipInterpolator(x_k, y_k)

print("Calibration fitting utilities loaded; monotonicity is checked for each fold-specific fitted map.")


## Fit on calibration only; apply to operating

Smoothed Isotonic uses the Stage 1 full-calibration reference `K=10`. Its isotonic base is fit using calibration scores and labels only. Operating labels are not passed to either fit function.


In [ ]:
cal_y = cal_df['label'].to_numpy(dtype=np.int64)
cal_s = cal_df['raw_score'].to_numpy(dtype=np.float64)
oper_y = oper_df['label'].to_numpy(dtype=np.int64)
oper_s = oper_df['raw_score'].to_numpy(dtype=np.float64)

platt_a, platt_b = fit_platt_model(cal_s, cal_y)
iso_model = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip').fit(cal_s, cal_y)
K_REFERENCE = 10
pchip_model = fit_smoothed_pchip(cal_s, iso_model, k_knots=K_REFERENCE)

oper_platt = np.asarray(apply_platt_model(oper_s, platt_a, platt_b), dtype=np.float64)
oper_smooth = np.clip(np.asarray(pchip_model(oper_s), dtype=np.float64), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)
assert np.isfinite(oper_platt).all() and ((oper_platt >= 0) & (oper_platt <= 1)).all()
assert np.isfinite(oper_smooth).all() and ((oper_smooth >= 0) & (oper_smooth <= 1)).all()


## Operating metrics and threshold selection

Raw scores are reported at the existing fixed 0.5 reference threshold. Platt and Smoothed Isotonic each receive a separate threshold selected on operating labels by `select_threshold` (recording-level macro-F1 and contract tie-breaks). These selected thresholds are operating-set results, not calibrator-fitting inputs.


In [ ]:
def score_metrics(labels, scores):
    return {
        'log_loss': compute_bce_loss(labels, scores),
        'brier_score': compute_brier_score(labels, scores),
        'equal_width_ece': compute_equal_width_ece(labels, scores, n_bins=10)[0],
        'equal_mass_ece': compute_equal_mass_ece(labels, scores, n_bins=10)[0],
        'roc_auc': _compute_roc_auc(np.asarray(labels, dtype=np.int64), np.asarray(scores, dtype=np.float64)),
        'eer': interpolated_eer(labels, scores),
    }

raw_metrics = score_metrics(oper_y, oper_s)
platt_metrics = score_metrics(oper_y, oper_platt)
smooth_metrics = score_metrics(oper_y, oper_smooth)
raw_threshold = {'kind': 'threshold', 'threshold': 0.5, 'macro_f1': None, 'genuine_fpr': None, 'policy': 'fixed reference 0.5'}
threshold_rows = {}
for name, scores in [('platt', oper_platt), ('smoothed_isotonic', oper_smooth)]:
    rule, macro_f1, genuine_fpr = select_threshold(oper_y.tolist(), scores.tolist())
    threshold_rows[name] = {
        'kind': rule.kind, 'threshold': rule.threshold, 'macro_f1': macro_f1,
        'genuine_fpr': genuine_fpr, 'tie_break': 'contract: lower FPR, closest to 0.5, then smaller threshold',
    }
metrics = {
    'raw_at_0_5': {**raw_metrics, 'threshold': raw_threshold},
    'platt': {**platt_metrics, 'threshold': threshold_rows['platt']},
    'smoothed_isotonic': {**smooth_metrics, 'threshold': threshold_rows['smoothed_isotonic']},
}
comparison_df = pd.DataFrame([{'method': name, **{k:v for k,v in row.items() if k != 'threshold'},
                               'threshold': row['threshold'].get('threshold'),
                               'threshold_macro_f1': row['threshold'].get('macro_f1'),
                               'threshold_genuine_fpr': row['threshold'].get('genuine_fpr')}
                              for name, row in metrics.items()])
display(comparison_df)


## Reliability diagrams

Reliability bins use the Stage 1 equal-width and equal-mass implementations with 10 bins. The operating labels are used only to summarize observed frequencies in these plots and metrics.


In [ ]:
oper_scores = {'Raw': oper_s, 'Platt': oper_platt, 'Smoothed Isotonic': oper_smooth}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
reliability_rows = []
for ax, (label, scores) in zip(axes, oper_scores.items()):
    ece, bins, n_valid = compute_equal_width_ece(oper_y, scores, n_bins=10)
    points = [b for b in bins if not b['is_empty']]
    ax.plot([0,1], [0,1], 'k--', linewidth=1)
    ax.plot([b['confidence'] for b in points], [b['accuracy'] for b in points], marker='o', label=f'ECE={ece:.4f}')
    ax.set_title(f'{label} (valid bins {n_valid}/10)')
    ax.set_xlabel('Mean predicted TTS probability')
    ax.grid(alpha=.25)
    ax.legend()
    for b in bins:
        reliability_rows.append({'method': label, **b})
axes[0].set_ylabel('Observed TTS frequency')
fig.suptitle('Operating partition reliability (N=1,040)')
fig.tight_layout()
plt.show()


## Reload verification and export

The Platt mapping is reloaded from its JSON coefficients. The PCHIP mapping is reconstructed from exported calibration-derived knot coordinates and values; both must reproduce operating predictions within absolute tolerance `1e-12`. Source prediction and checkpoint hashes are rechecked before evidence is finalized. A unique run directory is created; existing artifacts are never overwritten.


In [ ]:
# Serialize sufficient calibrator parameters for deterministic reload.
x_knots = np.unique(np.concatenate([[0.0], np.quantile(cal_s, np.linspace(0.0, 1.0, K_REFERENCE)), [1.0]]))
y_knots = iso_model.predict(x_knots)
assert np.all(np.diff(y_knots) >= -1e-12)
calibrator_payload = {
    'task': 'R3-N1 Stage 2 operational calibration',
    'source_calibration_path': str(CAL_FILE.relative_to(ROOT)),
    'source_calibration_sha256': EXPECTED['calibration'],
    'score_transform': {'logit': 'log(clip(p, epsilon, 1-epsilon)/(1-clip(p, epsilon, 1-epsilon)))', 'epsilon': DEFAULT_EPSILON},
    'platt': {'a': float(platt_a), 'b': float(platt_b), 'formula': 'sigmoid(a * logit(score) + b)'},
    'smoothed_isotonic': {'K': K_REFERENCE, 'x_knots': x_knots.tolist(), 'y_knots': y_knots.tolist(), 'interpolator': 'scipy PchipInterpolator', 'output_clip': [DEFAULT_EPSILON, 1.0-DEFAULT_EPSILON]},
}
reloaded_platt = apply_platt_model(oper_s, calibrator_payload['platt']['a'], calibrator_payload['platt']['b'])
reloaded_pchip = PchipInterpolator(np.asarray(calibrator_payload['smoothed_isotonic']['x_knots']), np.asarray(calibrator_payload['smoothed_isotonic']['y_knots']))
reloaded_smooth = np.clip(reloaded_pchip(oper_s), DEFAULT_EPSILON, 1.0-DEFAULT_EPSILON)
np.testing.assert_allclose(reloaded_platt, oper_platt, rtol=0, atol=1e-12)
np.testing.assert_allclose(reloaded_smooth, oper_smooth, rtol=0, atol=1e-12)

# Freeze plot and tabular evidence into a unique Stage 2 run directory.
run_id, out_dir = create_run(ROOT / 'runs/r3/r3_n1_operational_calibration')
plot_path = out_dir / 'operating_reliability.png'
fig.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.close(fig)
oper_export = oper_df.copy()
oper_export['calibrated_platt'] = oper_platt
oper_export['calibrated_smoothed_isotonic'] = oper_smooth
parquet_path = out_dir / 'operating_predictions.parquet'
csv_path = out_dir / 'operating_predictions.csv'
pq.write_table(pa.Table.from_pandas(oper_export, preserve_index=False), parquet_path)
oper_export.to_csv(csv_path, index=False)
metrics_path = out_dir / 'operating_metrics.json'
write_json(metrics_path, metrics)
comparison_path = out_dir / 'comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
reliability_path = out_dir / 'reliability_bins.csv'
pd.DataFrame(reliability_rows).to_csv(reliability_path, index=False)
calibrator_path = out_dir / 'calibrators.json'
write_json(calibrator_path, calibrator_payload)
threshold_path = out_dir / 'thresholds.json'
write_json(threshold_path, threshold_rows)

# Reload exported predictions and calibrator parameters, then verify reproducibility.
reloaded_export = pq.read_table(parquet_path).to_pandas()
assert len(reloaded_export) == 1040 and reloaded_export.recording_id.astype(str).tolist() == oper_df.recording_id.astype(str).tolist()
loaded_params = json.loads(calibrator_path.read_text(encoding='utf-8'))
np.testing.assert_allclose(apply_platt_model(oper_s, loaded_params['platt']['a'], loaded_params['platt']['b']), reloaded_export.calibrated_platt, rtol=0, atol=1e-12)
loaded_x = np.asarray(loaded_params['smoothed_isotonic']['x_knots'], dtype=np.float64)
loaded_y = np.asarray(loaded_params['smoothed_isotonic']['y_knots'], dtype=np.float64)
np.testing.assert_allclose(np.clip(PchipInterpolator(loaded_x, loaded_y)(oper_s), DEFAULT_EPSILON, 1-DEFAULT_EPSILON), reloaded_export.calibrated_smoothed_isotonic, rtol=0, atol=1e-12)
assert sha256(CAL_FILE) == EXPECTED['calibration'] and sha256(OPER_FILE) == EXPECTED['operating'] and sha256(CHECKPOINT_FILE) == EXPECTED['checkpoint']

artifact_names = ['calibrators.json','operating_predictions.parquet','operating_predictions.csv','operating_metrics.json','comparison.csv','reliability_bins.csv','thresholds.json','operating_reliability.png']
output_hashes = {name: sha256(out_dir / name) for name in artifact_names}
evidence_payload = {
    'task': 'R3-N1 Stage 2 operational calibration', 'status': 'complete', 'run_id': run_id,
    'created_utc': datetime.now(timezone.utc).isoformat(), 'source_run': str(SOURCE_DIR.relative_to(ROOT)),
    'source_hashes': {'calibration': EXPECTED['calibration'], 'operating': EXPECTED['operating'], 'selected_checkpoint': EXPECTED['checkpoint']},
    'selection': {'selected': 'lower_lr', 'selection_record_sha256': sha256(SELECTION_FILE), 'contract_sha256': sha256(CONTRACT_FILE)},
    'counts': {'calibration': len(cal_df), 'operating': len(oper_df)},
    'boundaries': {'calibration_used_for_fit': True, 'operating_used_for_fit': False, 'operating_used_for_threshold_and_evaluation': True, 'test_partition_loaded': False, 'bigru_retrained_or_loaded': False, 'checkpoint_hash_rechecked_after': True, 'source_predictions_modified': False, 'calibration_operating_ids_disjoint': True},
    'smoothed_isotonic_reference_K': K_REFERENCE, 'metrics': metrics, 'output_sha256': output_hashes,
}
evidence_path = out_dir / 'evidence.json'
write_json(evidence_path, evidence_payload)
manifest = make_manifest(out_dir, artifact_names + ['evidence.json'])
write_json(out_dir / 'artifact_manifest.json', manifest)
print(f'Operational calibration artifacts written: {out_dir}')
print(f'Artifact manifest SHA-256: {sha256(out_dir / "artifact_manifest.json")}')
